## 문서 파싱 기능 제공 ai_parse()
지원 포맷 
* PDF 
* JPG/JPEG
* PNG
* DOC/DOCX
* PPT/PPTX

ai_parse() 성능 참조

https://www.databricks.com/blog/pdfs-production-announcing-state-art-document-intelligence-databricks

In [0]:
# 샘플 PDF 파일을 볼륨에 저장 


In [0]:
%sql

-- To-Do: 개별 실습 환경에 맞게 카탈로그, 스키마, 볼륨 경로 변경 
CREATE or REPLACE TABLE hpark_demos.ski_agent_workshop.doc_parsed_tab 
AS SELECT
  path,
  ai_parse_document(
    content,
    map(
      'version', '2.0',
      'imageOutputPath', '/Volumes/hpark_demos/ski_agent_workshop/raw_data/parsed_images/',
      'descriptionElementTypes', '*'
    )
  ) as parsed_doc
FROM READ_FILES('/Volumes/hpark_demos/ski_agent_workshop/raw_data/*.pdf', format => 'binaryFile');



In [0]:
%sql
SELECT * FROM hpark_demos.ski_agent_workshop.doc_parsed_tab LIMIT 10

In [0]:
%sql
-- Vector Search용 소스 테이블 생성
CREATE OR REPLACE TABLE hpark_demos.ski_agent_workshop.doc_for_vector_search
TBLPROPERTIES (delta.enableChangeDataFeed = true) AS
WITH json_parsed AS (
  SELECT 
    path,
    explode(from_json(to_json(parsed_doc:document.elements), 'ARRAY<STRUCT<id:INT, content:STRING, bbox:ARRAY<STRUCT<page_id:INT, coord:ARRAY<INT>>>, type:STRING>>')) as element
  FROM hpark_demos.ski_agent_workshop.doc_parsed_tab
)
SELECT 
    path,
    element.content AS chunk_text,
    element.type AS element_type,
    element.bbox[0].page_id AS element_page_id,
    md5(concat(path, element.id)) as chunk_id
FROM json_parsed 
WHERE element.content IS NOT NULL AND element.type = 'text';

In [0]:
%sql
SELECT * FROM hpark_demos.ski_agent_workshop.doc_for_vector_search LIMIT 10

In [0]:
# Vector Search endpoint 생성

In [0]:
# Vector Search Index 생성 

In [0]:
%sql
SELECT * FROM 